# Train Khatib v8

Trains the NNUE on Lichess's Stockfish-evaluation database.

**First:** Runtime -> Change runtime type -> **T4 GPU**, then Save.

Run the cells in order. Free Colab can disconnect at any time, so every epoch
is saved to Google Drive -- if it drops, just re-run the training cell and it
picks up where it left off.

In [ ]:
# 1. Confirm a GPU is attached.
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
print('torch sees GPU:', torch.cuda.is_available())

In [ ]:
# 2. Mount Drive (click through the popup) so checkpoints survive a drop.
from google.colab import drive
drive.mount('/content/drive')

import os
CKPT = '/content/drive/MyDrive/khatib'
os.makedirs(CKPT, exist_ok=True)
print('checkpoints ->', CKPT)

In [ ]:
# 3. Fetch the trainer.
!rm -rf /content/khatib
!git clone -q https://github.com/Nesbesss/khatib-chess /content/khatib
!pip install -q zstandard
print('ok')

In [ ]:
# 4. Download positions from Lichess (~10 min). Streams and filters;
#    the 21 GB archive is never stored.
POSITIONS = 95_000_000
DATA = '/content/lichess_evals.txt'

import os
have = os.path.exists(DATA) and os.path.getsize(DATA) > 1e9
print('already downloaded' if have else 'downloading...')

if not have:
    !cd /content/khatib && python3 -u trainer/convert_lichess_evals.py --out $DATA --limit $POSITIONS

print('size:', round(os.path.getsize(DATA) / 1e9, 2), 'GB')

In [ ]:
# 5. Train. This is the long cell -- roughly 6-11 hours on a T4.
#    If the session drops, just run this cell again: it resumes from Drive.
#    If it dies while loading, that is the RAM limit -- set LIMIT to 30_000_000.
EPOCHS = 20
LIMIT = 60_000_000

import os
os.environ['CHESS_HIDDEN'] = '2048'

!cd /content/khatib && python3 -u trainer/train.py --data $DATA --limit $LIMIT --out $CKPT/v8.nnue --epochs $EPOCHS --batch 16384 --lr 1e-3 --lambda 0.7 --checkpoint-every 1 --resume-state $CKPT/state.pt

In [ ]:
# 6. Check the result. The engine expects exactly 26,219,024 bytes.
import os, glob
for p in sorted(glob.glob(CKPT + '/v8.nnue*')):
    print(f'{os.path.basename(p):22} {os.path.getsize(p):,} bytes')
print()
print('Download v8.nnue from Drive and send it over for testing against v7.')